In [1]:
import pandas as pd

lab_features_df = pd.read_parquet(
    "../data/processed/lab_features_first24h.parquet"
)
lab_features_df.head()

   ICUSTAY_ID  Anion Gap_first  ...  pO2_first  pO2_change
0      200001             14.0  ...       79.0        35.0
1      200003             21.0  ...       88.0        75.0
2      200006             12.0  ...        NaN         NaN
3      200007             17.0  ...        NaN         NaN
4      200009             11.0  ...      437.0      -343.0

[5 rows x 39 columns]


In [2]:
from pathlib import Path

data_path = Path(
    "/mnt/c/Users/vetts/Downloads/MimicIII/mimic-iii-clinical-database-1.4"
)

In [30]:
chartevents_sample_df = pd.read_csv(
    data_path / "CHARTEVENTS.csv" / "CHARTEVENTS.csv",
    nrows=5
)

chartevents_sample_df.columns

Index(['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME',
       'STORETIME', 'CGID', 'VALUE', 'VALUENUM', 'VALUEUOM', 'WARNING',
       'ERROR', 'RESULTSTATUS', 'STOPPED'],
      dtype='str')

In [31]:
clinical_items_df = pd.read_csv(
    data_path / "D_ITEMS.csv" / "D_ITEMS.csv"
)

clinical_items_df[["ITEMID", "LABEL", "CATEGORY", "LINKSTO"]].head()

,ITEMID,LABEL,CATEGORY,LINKSTO
0,497,Patient controlled analgesia (PCA) [Inject],NaN,chartevents
1,498,PCA Lockout (Min),NaN,chartevents
2,499,PCA Medication,NaN,chartevents
3,500,PCA Total Dose,NaN,chartevents
4,501,PCV Exh Vt (Obser),NaN,chartevents


In [32]:
clinical_items_df["LABEL"].unique()

<ArrowStringArray>
['Patient controlled analgesia (PCA) [Inject]',
                           'PCA Lockout (Min)',
                              'PCA Medication',
                              'PCA Total Dose',
                          'PCV Exh Vt (Obser)',
                                   'Allergy 2',
                                         'Ext',
                                   'Allergy 3',
                              'blood cultures',
                                  'trach care',
 ...
                     'CreatinineApacheIIValue',
                             'DswfApacheScore',
                           'FiO2ApacheIIValue',
                            'GcsApacheIIScore',
                         'GCSEyeApacheIIValue',
                       'GCSMotorApacheIIValue',
                      'GCSVerbalApacheIIValue',
                           'HCO3ApacheIIValue',
                                   'HCO3Score',
                     'HematocritApacheIIScore']
Length: 11847, d

In [33]:
RUN_FULL_FREQUENCY_SCAN = False

# Optional historical exploration; unnecessary for selected-vital extraction.
if RUN_FULL_FREQUENCY_SCAN:
    from collections import defaultdict

    item_icu_sets = defaultdict(set)

    chartevents_path = data_path / "CHARTEVENTS.csv" / "CHARTEVENTS.csv"

    for chunk in pd.read_csv(
        chartevents_path,
        usecols=["ICUSTAY_ID", "ITEMID"],
        chunksize=500_000
    ):
        chunk = chunk.dropna(subset=["ICUSTAY_ID"])

        for itemid, icu_ids in chunk.groupby("ITEMID")["ICUSTAY_ID"]:
            item_icu_sets[itemid].update(icu_ids.unique())

    chartevent_frequency = pd.DataFrame({
        "ITEMID": item_icu_sets.keys(),
        "icu_stay_count": [len(icu_ids) for icu_ids in item_icu_sets.values()]
    })

    top_25_chartevents = (
        chartevent_frequency
        .sort_values("icu_stay_count", ascending=False)
        .head(25)
    )

    top_25_chartevents


In [34]:
# Optional historical exploration; unnecessary for selected-vital extraction.
if RUN_FULL_FREQUENCY_SCAN:
    top_25_chartevents_named = top_25_chartevents.merge(
        clinical_items_df[["ITEMID", "LABEL", "CATEGORY"]],
        on="ITEMID",
        how="left"
    )

    top_25_chartevents_named


In [35]:
# Optional historical exploration; unnecessary for selected-vital extraction.
if RUN_FULL_FREQUENCY_SCAN:
    total_icu_stays = lab_features_df["ICUSTAY_ID"].nunique()

    chartevents_50pct = chartevent_frequency[
        chartevent_frequency["icu_stay_count"] >= total_icu_stays * 0.50
    ].copy()

    chartevents_50pct = chartevents_50pct.merge(
        clinical_items_df[["ITEMID", "LABEL", "CATEGORY"]],
        on="ITEMID",
        how="left"
    )

    chartevents_50pct.sort_values(
        "icu_stay_count",
        ascending=False
    )


In [36]:
selected_chart_itemids = {
    "Heart Rate": 211,
    "Respiratory Rate": 618,
    "SpO2": 646,
    "GCS Total": 198
}

In [37]:
clinical_items_df[
    clinical_items_df["LABEL"].str.contains(
        "Blood Pressure",
        case=False,
        na=False
    )
][["ITEMID", "LABEL"]]

,ITEMID,LABEL
9314,224167,Manual Blood Pressure Systolic Left
9441,227242,Manual Blood Pressure Diastolic Right
9442,227243,Manual Blood Pressure Systolic Right
9578,223751,Non-Invasive Blood Pressure Alarm - High
10261,227537,ART Blood Pressure Alarm - High
10262,227538,ART Blood Pressure Alarm - Low
10263,227539,ART Blood Pressure Alarm Source
11323,224643,Manual Blood Pressure Diastolic Left
11502,220050,Arterial Blood Pressure systolic
11503,220051,Arterial Blood Pressure diastolic


In [38]:
# Optional historical exploration; unnecessary for selected-vital extraction.
if RUN_FULL_FREQUENCY_SCAN:
    bp_itemids = [220050, 220051, 220052, 220179, 220180, 220181]

    bp_frequency = chartevent_frequency[
        chartevent_frequency["ITEMID"].isin(bp_itemids)
    ].merge(
        clinical_items_df[["ITEMID", "LABEL"]],
        on="ITEMID",
        how="left"
    )

    bp_frequency.sort_values("icu_stay_count", ascending=False)


In [39]:
vital_keywords = {
    "Heart Rate": r"heart rate",
    "Respiratory Rate": r"respiratory rate",
    "SpO2": r"spo2|oxygen saturation",
    "Systolic BP": r"non.?invasive.*systolic|nbp.*systolic",
    "Diastolic BP": r"non.?invasive.*diastolic|nbp.*diastolic",
    "Mean BP": r"non.?invasive.*mean|nbp.*mean",
    "Temperature": r"temperature",
    "GCS Total": r"gcs total"
}

for vital, pattern in vital_keywords.items():
    print(f"\n--- {vital} ---")
    display(
        clinical_items_df[
            clinical_items_df["LABEL"].str.contains(
                pattern,
                case=False,
                na=False,
                regex=True
            )
        ][["ITEMID", "LABEL", "DBSOURCE", "LINKSTO"]]
    )


--- Heart Rate ---


,ITEMID,LABEL,DBSOURCE,LINKSTO
475,211,Heart Rate,carevue,chartevents
1897,3494,Lowest Heart Rate,carevue,chartevents
11498,220045,Heart Rate,metavision,chartevents
11499,220046,Heart rate Alarm - High,metavision,chartevents
11500,220047,Heart Rate Alarm - Low,metavision,chartevents



--- Respiratory Rate ---


,ITEMID,LABEL,DBSOURCE,LINKSTO
263,618,Respiratory Rate,carevue,chartevents
1395,619,Respiratory Rate Set,carevue,chartevents
10052,224688,Respiratory Rate (Set),metavision,chartevents
10053,224689,Respiratory Rate (spontaneous),metavision,chartevents
10054,224690,Respiratory Rate (Total),metavision,chartevents
11524,220210,Respiratory Rate,metavision,chartevents



--- SpO2 ---


...

,ITEMID,LABEL,DBSOURCE,LINKSTO
683,456,NBP Mean,carevue,chartevents
11522,220181,Non Invasive Blood Pressure mean,metavision,chartevents



--- Temperature ---


,ITEMID,LABEL,DBSOURCE,LINKSTO
236,591,RLE [Temperature],carevue,chartevents
242,597,RUE [Temperature],carevue,chartevents
1417,645,Skin [Temperature],carevue,chartevents
1446,676,Temperature C,carevue,chartevents
1447,677,Temperature C (calc),carevue,chartevents
1448,678,Temperature F,carevue,chartevents
1449,679,Temperature F (calc),carevue,chartevents
4813,8537,"Temp/Iso/Warmer [Temperature, degrees C]",carevue,chartevents
9306,224027,Skin Temperature,metavision,chartevents
9415,227054,TemperatureF_ApacheIV,metavision,chartevents



--- GCS Total ---


,ITEMID,LABEL,DBSOURCE,LINKSTO
462,198,GCS Total,carevue,chartevents


In [40]:
vital_itemids = {
    "Heart Rate": [211, 220045],
    "Respiratory Rate": [618, 220210],
    "SpO2": [646, 220277],
    "Systolic BP": [455, 220179],
    "Diastolic BP": [8441, 220180],
    "Mean BP": [456, 220181],
    "Temperature C": [676, 223762],
    "Temperature F": [678, 223761],
    "GCS Total": [198]
}

vital_itemids

{'Heart Rate': [211, 220045],
 'Respiratory Rate': [618, 220210],
 'SpO2': [646, 220277],
 'Systolic BP': [455, 220179],
 'Diastolic BP': [8441, 220180],
 'Mean BP': [456, 220181],
 'Temperature C': [676, 223762],
 'Temperature F': [678, 223761],
 'GCS Total': [198]}

In [41]:
# Run once; subsequent runs reuse the validated Parquet cache.
# The extractor reads the literal vital_itemids mapping saved in this notebook.
import sys
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "scripts"))
from extract_vitals import run

extraction_report = run()
extraction_report


{'signature': {'inputs': {'CHARTEVENTS': [35307895134, 1788516163270726400],
   'ICUSTAYS': [6357077, 1788516165910647500],
   'PATIENTS': [2628900, 1788516220014278800]},
  'items': {'211': 'Heart Rate',
   '220045': 'Heart Rate',
   '618': 'Respiratory Rate',
   '220210': 'Respiratory Rate',
   '646': 'SpO2',
   '220277': 'SpO2',
   '455': 'Systolic BP',
   '220179': 'Systolic BP',
   '8441': 'Diastolic BP',
   '220180': 'Diastolic BP',
   '456': 'Mean BP',
   '220181': 'Mean BP',
   '676': 'Temperature C',
   '223762': 'Temperature C',
   '678': 'Temperature F',
   '223761': 'Temperature F',
   '198': 'GCS Total'},
  'version': 1},
 'source_rows': 330712483,
 'selected_item_rows': 31432046,
 'cohort_event_rows': 28754992,
 'missing_charttime_rows': 0,
 'output_rows': 7031935,
 'cohort_stays': 45253,
 'stays_with_vitals': 44549,
 'stays_without_vitals': 704,
 'elapsed_seconds': 937.8,
 'window': 'INTIME <= CHARTTIME < INTIME + 24 hours',
 'policy': 'Strict CSV parsing. Raw units, nul

## Reusable first-24-hour vital events

The extractor uses strict PyArrow streaming CSV parsing in 1 MiB blocks. It joins on SUBJECT_ID, HADM_ID and ICUSTAY_ID, using the original adult cohort with LOS >= 1 and nonmissing OUTTIME (45,253 stays). The window is INTIME <= CHARTTIME < INTIME + 24 hours. The 44,626-row lab table is not used to restrict this cohort.

Outputs: data/processed/vitals_first24h.parquet, adult_icu_cohort_first24h.parquet and vitals_first24h.json. Values, temperature units, missing values and ERROR flags are preserved for later cleaning; these are measurement events, not final model features. STORETIME is retained but does not filter the requested CHARTTIME window.

The saved notebook contained ModuleNotFoundError for duckdb, but no IndexError traceback. A strict streaming trial encountered an OS allocation error while WSL had only about 731 MiB available. This establishes memory pressure, not the exact cause of the reported pandas IndexError. No malformed CSV rows are silently skipped. A failed run leaves a .partial file and never publishes it as a completed cache.


In [16]:
vitals_df = pd.read_parquet(
    "../data/processed/vitals_first24h.parquet",
    columns=["ICUSTAY_ID", "CHARTTIME", "VITAL", "VALUENUM", "VALUEUOM", "ERROR"]
)
vitals_df.head()


   ICUSTAY_ID           CHARTTIME             VITAL  VALUENUM  VALUEUOM ERROR
0      241249 2134-05-12 13:00:00        Heart Rate      86.0       bpm     0
1      241249 2134-05-12 13:00:00       Systolic BP     137.0      mmHg     0
2      241249 2134-05-12 13:00:00      Diastolic BP      72.0      mmHg     0
3      241249 2134-05-12 13:00:00           Mean BP      84.0      mmHg     0
4      241249 2134-05-12 13:00:00  Respiratory Rate      21.0  insp/min     0


In [17]:
vitals_df["CHARTTIME"].dtype

datetime64[us]


In [18]:
from pathlib import Path
import sys
_quality_root = Path.cwd().resolve()
if _quality_root.name == 'notebooks':
    _quality_root = _quality_root.parent
sys.path.insert(0, str(_quality_root / 'scripts'))
from data_quality import clean_vitals
vitals_df, vital_quality_audit = clean_vitals(vitals_df)
print(vital_quality_audit.to_string(index=False))


           vital  range_invalid_records  error_flag_records  invalid_union
    Diastolic BP                    523                 367            888
       GCS Total                      0                   0              0
      Heart Rate                    155                  92            244
         Mean BP                    185                 292            476
Respiratory Rate                   6280                 169           6408
            SpO2                    120                 427            544
     Systolic BP                    672                 376           1045
   Temperature C                    275                 137            310
   Temperature F                    473                 398            727


In [19]:
vitals_df["VITAL"].unique()

<ArrowStringArray>
[      'Heart Rate',      'Systolic BP',     'Diastolic BP',
          'Mean BP', 'Respiratory Rate',             'SpO2',
      'Temperature',        'GCS Total']
Length: 8, dtype: str


In [20]:
vitals_sorted_df = vitals_df.sort_values(
    ["ICUSTAY_ID", "VITAL", "CHARTTIME"]
)

vital_summary_df = (
    vitals_sorted_df
    .groupby(["ICUSTAY_ID", "VITAL"])["VALUENUM"]
    .agg(
        first="first",
        last="last",
        min="min",
        max="max"
    )
    .reset_index()
)

vital_summary_df.head()

   ICUSTAY_ID             VITAL  first   last   min    max
0      200001      Diastolic BP   65.0   55.0  49.0   68.0
1      200001        Heart Rate  114.0   91.0  88.0  134.0
2      200001           Mean BP   77.0   64.0  60.0   79.0
3      200001  Respiratory Rate   22.0   19.0  15.0   32.0
4      200001              SpO2   94.0  100.0  94.0  100.0


In [21]:
vital_summary_df["VITAL"].unique()

<ArrowStringArray>
[    'Diastolic BP',       'Heart Rate',          'Mean BP',
 'Respiratory Rate',             'SpO2',      'Systolic BP',
      'Temperature',        'GCS Total']
Length: 8, dtype: str


In [22]:
vital_features_df = vital_summary_df.pivot(
    index="ICUSTAY_ID",
    columns="VITAL",
    values=["first", "last", "min", "max"]
)

vital_features_df.head()

                  first                       ...    max                        
VITAL      Diastolic BP GCS Total Heart Rate  ...   SpO2 Systolic BP Temperature
ICUSTAY_ID                                    ...                               
200001             65.0       NaN      114.0  ...  100.0       116.0   37.666667
200003             49.0      15.0      119.0  ...   98.0       115.0   38.999998
200006             87.0      15.0       84.0  ...  100.0       118.0   37.666668
200007             71.0      15.0       90.0  ...   97.0       144.0   37.611109
200009             58.0       3.0       92.0  ...  100.0        89.0   38.299999

[5 rows x 32 columns]


In [23]:
vital_features_df.columns = [
    f"{vital}_{stat}"
    for stat, vital in vital_features_df.columns
]

vital_features_df = vital_features_df.reset_index()

vital_features_df.head()

   ICUSTAY_ID  Diastolic BP_first  ...  Systolic BP_max  Temperature_max
0      200001                65.0  ...            116.0        37.666667
1      200003                49.0  ...            115.0        38.999998
2      200006                87.0  ...            118.0        37.666668
3      200007                71.0  ...            144.0        37.611109
4      200009                58.0  ...             89.0        38.299999

[5 rows x 33 columns]


In [24]:
missing_vitals_df = pd.DataFrame({
    "missing_count": vital_features_df.isna().sum(),
    "missing_percent": vital_features_df.isna().mean() * 100
})

missing_vitals_df = missing_vitals_df.sort_values(
    "missing_percent",
    ascending=False
)

missing_vitals_df

                        missing_count  missing_percent
GCS Total_first                 19570        43.929157
GCS Total_max                   19570        43.929157
GCS Total_last                  19570        43.929157
GCS Total_min                   19570        43.929157
Mean BP_min                      6021        13.515455
Mean BP_last                     6021        13.515455
Mean BP_max                      6021        13.515455
Mean BP_first                    6021        13.515455
Diastolic BP_first               5996        13.459337
Diastolic BP_max                 5996        13.459337
Diastolic BP_last                5996        13.459337
Diastolic BP_min                 5996        13.459337
Systolic BP_min                  5977        13.416687
Systolic BP_last                 5977        13.416687
Systolic BP_max                  5977        13.416687
Systolic BP_first                5977        13.416687
Temperature_max                   745         1.672316
Temperatur

## Vital Signs Feature Engineering

Vital sign records from the first 24 hours of each ICU stay were processed.

- Fahrenheit temperatures were converted to Celsius and merged under a single `Temperature` variable.
- For each ICU stay and each vital sign, `first`, `last`, `min`, and `max` values were calculated.
- The data was pivoted so that each ICU stay represents one row.
- Missing value rates were checked for all vital features.
- GCS had the highest missingness (~44%) but was kept because of its clinical importance.

In [26]:
vital_features_df.to_parquet(
    "../data/processed/vital_features_first24h.parquet",
    index=False
)

In [ ]:
import extract_vitals
import inspect

print(extract_vitals.__file__)
print(inspect.getsource(extract_vitals.run).splitlines()[1])

In [ ]:
import importlib
import extract_vitals

importlib.reload(extract_vitals)

extraction_report = extract_vitals.run()
extraction_report

In [ ]:
vital_features_df.to_parquet(
    "../data/processed/vital_features_first24h.parquet",
    index=False
)

In [ ]:
vitals_df.loc[
    (vitals_df["VITAL"] == "Temperature") &
    (vitals_df["VALUENUM"] < 0),
    ["ICUSTAY_ID", "CHARTTIME", "VALUENUM", "VALUEUOM"]
].head(30)

## Source-level quality review (2026-09-12)
Quality rules are in `scripts/data_quality.py` and are applied before aggregation. No features or cohort IDs are removed. Invalid measurement values become missing; no median imputation or percentile clipping is performed. Negative changes and negative anion gaps are not automatically invalid.
Vital bounds follow [MIMIC-code](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/firstday/vitals_first_day.sql): HR (0,300), RR (0,70), SBP (0,400), DBP/MAP (0,300), SpO2 (0,100], Fahrenheit (70,120) before conversion, Celsius (10,50), and integer GCS [3,15]. These are broad quality screens, not normal ranges. Error-flagged vital values are excluded.
Lab: nonfinite values and nonpositive hemoglobin/INR measurements are invalidated before first/last/change calculation. Other rare lab extremes remain flagged for review. Urine: reject negative, nonfinite and grossly implausible individual entries above 100,000 mL. The prior ICU 223940 correction remains 1,200 mL. The 47,050 mL entry is retained for review; EBL is unchanged.
